# Agentic search systems

This notebook walks through the ideas implemented in `agentic-search/packages/bcp-agent`: a **planner → executor (search tools) → evaluator → final answer** loop over a fixed corpus, backed by Chroma.

**Orchestration:** [LangGraph’s functional API](https://docs.langchain.com/oss/python/langgraph/functional-api) (`@task`, `@entrypoint`) drives the loop; **LangChain** [`create_agent`](https://docs.langchain.com/oss/python/langchain/agents) runs the per-step retrieval ReAct loop so we do not hand-roll tool calling.

**What you need:** Python 3.10+, `OPENAI_API_KEY`. The code stays short on purpose—no production-style error handling.

## BrowseComp-Plus and architecture

BrowseComp-Plus is based on OpenAI's BrowseComp benchmark. It is intentionally hard: many queries require multiple rounds of retrieval plus reasoning, so it is a strong dataset for evaluating agentic search design choices (retrieval quality, reasoning strategy, model selection, and tool behavior).

Each query includes:
- **Gold docs**: required to compile the final correct answer.
- **Evidence docs**: needed for reasoning, even if they do not directly state the final answer. Gold docs are a subset of evidence docs.
- **Negative docs**: distractors that force the agent to separate relevant from irrelevant context.

1. **Planner** — LLM turns the user question into a list of **steps** (see `prompts.ts` `generatePlan`). Each step has an id, title, optional `parents` for dependencies, and a description (`schemas.ts` extends the base step with `description`).
2. **Executor** — For each pending step, the LLM calls **tools** until it has enough context (`base-executor.ts`). Here those tools mirror `SemanticSearchTool`, `LexicalSearchTool`, and `HybridSearchTool` in `src/tools/`.
3. **Step outcome** — After tools run, the LLM **finalizes** the step: summary, document ids as **evidence**, optional **candidate answers** (`outcomeSchema` in `schemas.ts`).
4. **Plan evaluation** — The LLM chooses **continue**, **finalize** (stop early), or **override** the remaining plan with new steps (`evaluatePlanSystemPrompt` in `prompts.ts`).
5. **Final answer** — Grounded answer with **evidence**, **reason**, and **confidence** (`answerSchema` in `schemas.ts`).

The reference agent loads the **browse-comp-plus** collection and excludes rows where `query` is true (`chroma.ts`, `semantic-search.ts`). Below we use a tiny in-memory collection with the same metadata pattern.

**LangGraph:** each LLM-heavy chunk (plan, run retrieval agent, structured outcome, plan evaluation, final answer) is a `@task`; the `@entrypoint` function is plain Python control flow (`while`, `if`) that calls `.result()` on those tasks—same persistence/streaming hooks as other LangGraph workflows.

In [ ]:
%pip install -qqq chromadb datasets langchain langchain-openai langgraph

In [ ]:
import json
import re

import chromadb
from datasets import load_dataset
from langchain_openai import ChatOpenAI

MODEL = "gpt-4o-mini"
chroma = chromadb.EphemeralClient()
llm = ChatOpenAI(model=MODEL, temperature=0)

# BrowseComp-Plus datasets from Hugging Face (per repo README)
queries_ds = load_dataset("Tevatron/browsecomp-plus", split="train")
corpus_ds = load_dataset("Tevatron/browsecomp-plus-corpus", split="train")
print("queries:", len(queries_ds), "corpus:", len(corpus_ds))

## 1. Corpus in Chroma

We now use BrowseComp-Plus data directly:
- `Tevatron/browsecomp-plus` for query records.
- `Tevatron/browsecomp-plus-corpus` for corpus documents.

We ingest a configurable subset of corpus docs into Chroma with metadata `{doc_id, query: false}` so the tool filter pattern (`where: { query: { $ne: true } }`) stays aligned with the TypeScript agent.

To keep notebook runtime manageable, we index a subset by default (`max_docs=2000`). Increase it for better retrieval coverage.

In [ ]:
def tokenize(text: str) -> set[str]:
    return {t for t in re.findall(r"[a-z0-9]+", text.lower()) if len(t) > 1}


def _pick_first(row: dict, keys: list[str], default=""):
    for k in keys:
        if k in row and row[k] is not None:
            return row[k]
    return default


def build_browsecomp_collection(max_docs: int = 2000):
    col = chroma.get_or_create_collection(
        name="browsecomp-plus-corpus",
        metadata={"hnsw:space": "cosine"},
    )
    if col.count() > 0:
        return col

    ids, docs, metas = [], [], []
    for i, row in enumerate(corpus_ds.select(range(min(max_docs, len(corpus_ds))))):
        doc_id = str(_pick_first(row, ["doc_id", "id", "document_id"], default=f"doc-{i}"))
        text = _pick_first(row, ["text", "contents", "content", "document"], default="")
        ids.append(f"corpus-{i}")
        docs.append(text)
        metas.append({"doc_id": doc_id, "query": False})

    col.add(ids=ids, documents=docs, metadatas=metas)
    return col


collection = build_browsecomp_collection(max_docs=2000)
print("indexed_docs:", collection.count())

## 2. Three search tools (mirror `src/tools/`)

| Tool | Reference file | Idea |
|------|----------------|------|
| Semantic | `semantic-search.ts` | Dense embedding similarity via `collection.query` |
| Lexical | `lexical-search.ts` | Sparse / keyword-style retrieval |
| Hybrid | `hybrid-search.ts` | RRF over dense + sparse ranks |

Each tool returns **records** `{id, docId, document}` like `chroma-tool.ts` / `utils.ts` `processSearchResults`.

In [ ]:
CORPUS_FILTER = {"query": {"$ne": True}}


def rows_to_records(rows) -> list[dict]:
    out = []
    for r in rows:
        out.append(
            {
                "id": r["id"],
                "docId": (r.get("metadata") or {}).get("doc_id", "?"),
                "document": r.get("document") or "",
            }
        )
    return out


def semantic_search(collection, query: str, n: int = 5) -> list[dict]:
    res = collection.query(
        query_texts=[query],
        n_results=n,
        where=CORPUS_FILTER,
    )
    rows = [{"id": i, "metadata": m, "document": d} for i, m, d in zip(res["ids"][0], res["metadatas"][0], res["documents"][0])]
    return rows_to_records(rows)


def lexical_search(collection, query: str, n: int = 5) -> list[dict]:
    q_tokens = tokenize(query)
    res = collection.get(where=CORPUS_FILTER, include=["documents", "metadatas"])
    scored = []
    for i, doc in enumerate(res["documents"]):
        score = len(q_tokens & tokenize(doc or ""))
        scored.append((score, i))
    scored.sort(key=lambda x: (-x[0], x[1]))
    rows = []
    for _, idx in scored[:n]:
        rows.append(
            {
                "id": res["ids"][idx],
                "metadata": res["metadatas"][idx],
                "document": res["documents"][idx],
            }
        )
    return rows_to_records(rows)


def rrf_merge(
    ranked_lists: list[list[str]],
    weights: list[float],
    k: int = 60,
    limit: int = 5,
) -> list[str]:
    scores: dict[str, float] = {}
    for ranks, w in zip(ranked_lists, weights):
        for r, doc_id in enumerate(ranks):
            scores[doc_id] = scores.get(doc_id, 0.0) + w / (k + r + 1)
    ordered = sorted(scores.keys(), key=lambda x: -scores[x])
    return ordered[:limit]


def hybrid_search(
    collection,
    dense_query: str,
    sparse_query: str,
    n: int = 5,
    dense_weight: float = 2.0,
    sparse_weight: float = 1.0,
) -> list[dict]:
    sem = semantic_search(collection, dense_query, n=20)
    lex = lexical_search(collection, sparse_query, n=20)
    merged_ids = rrf_merge(
        [[r["id"] for r in sem], [r["id"] for r in lex]],
        [dense_weight, sparse_weight],
        limit=n,
    )
    by_id = {r["id"]: r for r in sem + lex}
    return [by_id[i] for i in merged_ids if i in by_id]


def format_tool_result(records: list[dict]) -> str:
    if not records:
        return "No records found"
    parts = []
    for rec in records:
        parts.append(f"// Document {rec['id']}\n// Corpus document ID: {rec['docId']}\n{rec['document']}")
    return "\n\n".join(parts)


from langchain.tools import tool


@tool
def semantic_search_tool(query: str, num_results: int = 5, reason: str = "") -> str:
    """Dense-vector semantic search. Use for conceptually related passages."""
    recs = semantic_search(collection, query, n=num_results)
    return format_tool_result(recs)


@tool
def lexical_search_tool(query: str, num_results: int = 5, reason: str = "") -> str:
    """Keyword-style search for exact terms, names, and numbers."""
    recs = lexical_search(collection, query, n=num_results)
    return format_tool_result(recs)


@tool
def hybrid_search_tool(
    dense_query: str,
    sparse_query: str,
    num_results: int = 5,
    dense_weight: float = 2.0,
    sparse_weight: float = 1.0,
    reason: str = "",
) -> str:
    """Semantic + lexical via RRF. Use for constraint-heavy questions."""
    recs = hybrid_search(
        collection,
        dense_query,
        sparse_query,
        n=num_results,
        dense_weight=dense_weight,
        sparse_weight=sparse_weight,
    )
    return format_tool_result(recs)


SEARCH_TOOLS = [semantic_search_tool, lexical_search_tool, hybrid_search_tool]

## 3. LangGraph workflow + LangChain retrieval agent

- **`@task`** — One unit of work per LLM call (or agent run). Checkpoint-friendly; `.result()` blocks inside the entrypoint ([functional API](https://docs.langchain.com/oss/python/langgraph/functional-api)).
- **`@entrypoint`** — Plain Python loop: plan → for each step, run retrieval agent → structured outcome → plan evaluation → final answer. Matches `BaseAgent.run` without implementing tool loops yourself.
- **`create_agent`** — LangChain agent (LangGraph under the hood) runs the ReAct tool loop for each step (`prompts.ts` execute-step behavior).

Prompts mirror `prompts.ts`: plan decomposition, execute with tools, outcome with evidence, `continue` / `finalize` / `overridePlan`, final grounded answer (`schemas.ts`).

In [ ]:
from typing import Literal

from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langchain.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage
)
from pydantic import BaseModel, Field



class PlanStep(BaseModel):
    id: str
    title: str
    description: str = ""
    parents: list[str] = Field(default_factory=list)


class PlanOut(BaseModel):
    steps: list[PlanStep]


class StepOutcome(BaseModel):
    stepId: str
    summary: str
    evidence: list[str]
    candidateAnswers: list[str] | None = None


class PlanEval(BaseModel):
    decision: Literal["continue", "finalize", "overridePlan"]
    reason: str
    newSteps: list[PlanStep] | None = None


class FinalAnswer(BaseModel):
    answer: str
    reason: str
    evidence: list[str]
    confidence: float = Field(ge=0.0, le=1.0)


PLAN_SYSTEM = """You are a query planner for a multi-step search agent over a fixed corpus.
Produce steps (id, title, description, optional parents). Do not answer the question. Do not invent document ids."""

EXEC_SYSTEM = """You execute ONE step of a search plan using tools. Call tools until you have enough evidence, then respond with a brief summary (no more tool calls)."""

EVAL_SYSTEM = """Given the question, plan, and history, choose:
- continue: run remaining steps
- finalize: stop planning; enough evidence for final answer
- overridePlan: replace pending steps with new ones (provide newSteps)."""

FINAL_SYSTEM = """Produce the final answer from evidence only. Ground in cited chunk ids."""


def step_user_block(user_query: str, plan: list[dict], step: dict, history: list[dict]) -> str:
    lines = [
        f"Original question: {user_query}",
        "Plan:",
        *[f"  {s['id']}: {s['title']}" for s in plan],
        f"Current step: {step['id']}: {step['title']}\n{step.get('description', '')}",
    ]
    if history:
        lines.append("Prior outcomes:")
        for h in history:
            lines.append(json.dumps(h, indent=2))
    return "\n".join(lines)


def serialize_messages(msgs) -> list[dict]:
    out = []
    for m in msgs:
        role = getattr(m, "type", None) or ""
        content = m.content
        if not isinstance(content, str):
            content = str(content)
        out.append({"role": role, "content": content})
    return out


retrieval_agent = create_agent(
    model=llm,
    tools=SEARCH_TOOLS,
    system_prompt=EXEC_SYSTEM,
)


@task
def plan_generation_task(inp: dict) -> dict:
    user_query = inp["user_query"]
    max_plan_size = inp["max_plan_size"]
    planner = llm.with_structured_output(PlanOut)
    out = planner.invoke(
        [
            ("system", PLAN_SYSTEM),
            ("user", f"Max {max_plan_size} steps.\nQuestion: {user_query}"),
        ]
    )
    return {"steps": [s.model_dump() for s in out.steps]}


@task
def retrieval_step_task(ctx: dict) -> dict:
    text = step_user_block(ctx["user_query"], ctx["plan"], ctx["step"], ctx["history"])
    result = retrieval_agent.invoke(
        {"messages": [("user", text)]},
        {"recursion_limit": 40},
    )
    return {"messages": serialize_messages(result["messages"])}


@task
def outcome_task(ctx: dict) -> dict:
    step = ctx["step"]
    transcript = json.dumps(ctx["agent_messages"], indent=2)
    structured = llm.with_structured_output(StepOutcome).invoke(
        [
            (
                "system",
                "Finalize the search step. Evidence must be Chroma record ids from // Document lines.",
            ),
            (
                "user",
                f"Step {step['id']}: {step['title']}\nTranscript:\n{transcript}",
            ),
        ]
    )
    return structured.model_dump()


@task
def evaluate_plan_task(inp: dict) -> dict:
    structured = llm.with_structured_output(PlanEval).invoke(
        [
            ("system", EVAL_SYSTEM),
            (
                "user",
                json.dumps(
                    {
                        "question": inp["user_query"],
                        "plan": inp["plan"],
                        "history": inp["history"],
                    }
                ),
            ),
        ]
    )
    return structured.model_dump()


@task
def final_answer_task(inp: dict) -> dict:
    structured = llm.with_structured_output(FinalAnswer).invoke(
        [
            ("system", FINAL_SYSTEM),
            (
                "user",
                json.dumps({"question": inp["user_query"], "history": inp["history"]}),
            ),
        ]
    )
    return structured.model_dump()


checkpointer = InMemorySaver()


@entrypoint(checkpointer=checkpointer)
def agentic_workflow(inp: dict) -> dict:
    user_query = inp["user_query"]
    max_plan_size = inp.get("max_plan_size", 4)
    raw = plan_generation_task({"user_query": user_query, "max_plan_size": max_plan_size}).result()
    plan = list(raw["steps"])
    history: list[dict] = []
    idx = 0
    while idx < len(plan):
        step = plan[idx]
        ctx = {
            "user_query": user_query,
            "plan": plan,
            "step": step,
            "history": history,
        }
        ar = retrieval_step_task(ctx).result()
        oc = outcome_task({**ctx, "agent_messages": ar["messages"]}).result()
        history.append(oc)
        ev = evaluate_plan_task(
            {"user_query": user_query, "plan": plan, "history": history}
        ).result()
        if ev.get("decision") == "finalize":
            break
        if ev.get("decision") == "overridePlan" and ev.get("newSteps"):
            plan = plan[: idx + 1] + list(ev["newSteps"])
        idx += 1
    final = final_answer_task({"user_query": user_query, "history": history}).result()
    return {"plan": plan, "history": history, "final": final}

The cell above defines `agentic_workflow` and runs one query. Use the same `config` (`thread_id`) for `invoke` or `stream` so checkpointing is consistent. Raise `recursion_limit` in `retrieval_agent.invoke` if a step needs extra tool rounds.

In [ ]:
# for chunk in agentic_workflow.stream(
#     {"user_query": "What causes an aurora?"},
#     {"configurable": {"thread_id": str(uuid7())}},
# ):
#     print(chunk)

In [ ]:
def get_query_text_by_id(query_id: int | str) -> str:
    qid = str(query_id)
    for row in queries_ds:
        row_qid = str(_pick_first(row, ["query_id", "id", "qid"], default=""))
        if row_qid == qid:
            return _pick_first(row, ["query", "question", "text"], default="")
    return ""


query_770 = get_query_text_by_id(770)
print("Loaded query 770:\n", query_770)

_cfg = {"configurable": {"thread_id": str(uuid7())}}
result = agentic_workflow.invoke(
    {"user_query": query_770, "max_plan_size": 6},
    _cfg,
)
print(json.dumps(result, indent=2))

## Takeaways

- **bcp-agent** wires the same loop through `BrowseCompPlusAgent` → `BaseAgent.create` with `bcpAgentPrompts` and `searchToolsFactory(collection)`.
- **LangGraph** `@entrypoint` + `@task` gives you checkpointing and streaming without encoding the control flow as a graph; **LangChain** `create_agent` owns the tool loop for each step.
- **Schemas** here are Pydantic models aligned with `schemas.ts` (steps, outcomes, evaluation, final answer).
- This notebook now downloads **BrowseComp-Plus queries and corpus** via `datasets`, indexes a corpus subset into Chroma, and resolves **query 770** from dataset metadata.

Set `OPENAI_API_KEY` before running; increase `max_docs` and adjust `MODEL` / `recursion_limit` as needed.